# Churn baseline notebook
Diagnostic only. With 12 rows no result here is a valid performance estimate. Re-run on a time-based dataset before any model decision.

In [1]:
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score

PATH = 'customer-churn-training.csv'
df = pd.read_csv(PATH)
y = df['churned']
print(df.shape)
print(df.isna().sum().sum(), 'missing;', df.duplicated().sum(), 'duplicate rows;', df.customer_id.duplicated().sum(), 'duplicate ids')

(12, 7)
0 missing; 0 duplicate rows; 0 duplicate ids


## 1. Data checks

In [2]:
print(y.value_counts(normalize=True))
print(df.groupby('plan_type').churned.agg(['count','sum']))
print(df.groupby('churned')[['tenure_months','support_tickets','monthly_spend_inr','last_login_days']].mean().round(1))

churned
0    0.583333
1    0.416667
Name: proportion, dtype: float64
           count  sum
plan_type            
Basic          5    5
Pro            3    0
Standard       4    0
         tenure_months  support_tickets  monthly_spend_inr  last_login_days
churned                                                                    
0                 17.9              1.1             1127.6              3.7
1                  4.2              3.8              619.0             18.8


## 2. Baseline A: majority class

In [3]:
maj = int(y.mode()[0])
print('majority-class accuracy:', round((y==maj).mean(),3))

majority-class accuracy: 0.583


## 3. Baseline B: transparent rules
Thresholds were picked after viewing the data, so these are hypotheses, not validated rules.

In [4]:
def report(name, pred):
    print(name, '| acc', round((pred==y).mean(),3), '| precision', round(precision_score(y,pred),3), '| recall', round(recall_score(y,pred),3))
    print(confusion_matrix(y,pred))

report('plan == Basic', (df.plan_type=='Basic').astype(int))
report('login>=10 or tenure<=6', ((df.last_login_days>=10)|(df.tenure_months<=6)).astype(int))

plan == Basic | acc 1.0 | precision 1.0 | recall 1.0
[[7 0]
 [0 5]]
login>=10 or tenure<=6 | acc 1.0 | precision 1.0 | recall 1.0
[[7 0]
 [0 5]]


## 4. Baseline C: regularised logistic regression, leave-one-out (diagnostic)
Also run without `plan_type` to see whether behavioural features alone separate the classes.

In [5]:
def loo(cols, cat=True):
    X = df[cols]
    if cat: X = pd.get_dummies(X, drop_first=True)
    m = make_pipeline(StandardScaler(), LogisticRegression(C=0.5))
    return cross_val_predict(m, X, y, cv=LeaveOneOut())

num = ['tenure_months','support_tickets','monthly_spend_inr','last_login_days']
report('LR with plan', loo(num+['plan_type']))
report('LR numeric only', loo(num, cat=False))

LR with plan | acc 1.0 | precision 1.0 | recall 1.0
[[7 0]
 [0 5]]
LR numeric only | acc 0.917 | precision 0.833 | recall 1.0
[[6 1]
 [0 5]]


## 5. Reading the result
- Perfect scores on 12 rows indicate separability or leakage, not a deployable model.
- Next: rebuild features as of a snapshot date, observe churn in a later window, split by time, and add calibration, top-K precision/recall, bootstrap CIs and fairness slices by plan, tenure and spend band.
- Proceed to ML only if it beats Baseline B on the holdout by the margin agreed in the data card.